In [ ]:
import os

os.makedirs("../data_dedup", exist_ok=True)
os.makedirs("../outputs_dedup", exist_ok=True)

print("✅ Dossiers Version B créés !")

In [ ]:
import pandas as pd
import numpy as np

df_train_original = pd.read_csv("../data/train.csv")
df_test_original = pd.read_csv("../data/test.csv")

df_full_dedup = pd.concat([df_train_original, df_test_original]).drop_duplicates(subset="Emoji_Text").reset_index(drop=True)

print(f"Avant déduplication : {len(df_train_original) + len(df_test_original)} tweets")
print(f"Après déduplication : {len(df_full_dedup)} tweets")
print(f"Doublons retirés : {len(df_train_original) + len(df_test_original) - len(df_full_dedup)}")

print(f"\nDistribution du label après déduplication :")
print(df_full_dedup["label"].value_counts())
print(df_full_dedup["label"].value_counts(normalize=True) * 100)

In [ ]:
from sklearn.model_selection import train_test_split

df_train_dedup, df_test_dedup = train_test_split(
    df_full_dedup,
    test_size=0.2,
    stratify=df_full_dedup["label"],
    random_state=42
)

print(f"Train : {df_train_dedup.shape}")
print(f"Test  : {df_test_dedup.shape}")

print(f"\nDistribution Train :")
print(df_train_dedup["label"].value_counts(normalize=True) * 100)

print(f"\nDistribution Test :")
print(df_test_dedup["label"].value_counts(normalize=True) * 100)

# Sauvegarder dans le nouveau dossier data_dedup
df_train_dedup.to_csv("../data_dedup/train.csv", index=False)
df_test_dedup.to_csv("../data_dedup/test.csv", index=False)

print("\n✅ train.csv et test.csv (dédupliqués) sauvegardés dans data_dedup/ !")

In [ ]:
train_texts_dedup = set(df_train_dedup["Emoji_Text"].dropna())
test_texts_dedup = set(df_test_dedup["Emoji_Text"].dropna())
doublons_restants = train_texts_dedup.intersection(test_texts_dedup)
print(f"Doublons restants entre train et test : {len(doublons_restants)}")

In [ ]:
#Ré-extraire le vocabulaire discriminant sur le nouveau train dédupliqué

In [ ]:
from collections import Counter
import re

tweets_sarc_dedup = df_train_dedup[df_train_dedup["label"] == 1]["Emoji_Text"].tolist()
tweets_non_sarc_dedup = df_train_dedup[df_train_dedup["label"] == 0]["Emoji_Text"].tolist()

def tokenize_arabic(text):
    if not isinstance(text, str):
        return []
    words = re.findall(r'[\u0600-\u06FF]+', text)
    return [w for w in words if len(w) > 2]

counter_sarc_dedup = Counter()
counter_non_sarc_dedup = Counter()

for tweet in tweets_sarc_dedup:
    counter_sarc_dedup.update(tokenize_arabic(tweet))
for tweet in tweets_non_sarc_dedup:
    counter_non_sarc_dedup.update(tokenize_arabic(tweet))

total_sarc_dedup = sum(counter_sarc_dedup.values())
total_non_sarc_dedup = sum(counter_non_sarc_dedup.values())

discriminant_scores_dedup = {}
for word in counter_sarc_dedup:
    freq_sarc = counter_sarc_dedup[word] / total_sarc_dedup
    freq_non_sarc = (counter_non_sarc_dedup.get(word, 0) + 1) / total_non_sarc_dedup
    discriminant_scores_dedup[word] = freq_sarc / freq_non_sarc

vocab_discriminant_dedup = sorted(discriminant_scores_dedup.items(), key=lambda x: x[1], reverse=True)
vocab_sarc_moji_dedup = [word for word, score in vocab_discriminant_dedup[:100]]

print("✅ Vocabulaire discriminant (dédupliqué) extrait !")
print("\nTop 10 mots :")
for word, score in vocab_discriminant_dedup[:10]:
    print(f"  {word} : score = {score:.3f}")

In [ ]:
import emoji

def discriminant_vocab_score(text):
    if not isinstance(text, str):
        return 0
    words = tokenize_arabic(text)
    if len(words) == 0:
        return 0
    count = sum(1 for w in words if w in vocab_sarc_moji_dedup)
    return count / len(words)

def hashtag_count(text):
    if not isinstance(text, str):
        return 0
    return len(re.findall(r'#\w+', text))

def emoji_count_feature(text):
    if not isinstance(text, str):
        return 0
    return sum(1 for c in text if c in emoji.EMOJI_DATA)

interjections = ['آه', 'أوه', 'يا إلهي', 'واو', 'يا ربي', 'هههه', 'هاها', 'لول', 'lol',
                 'هيه', 'آآآ', 'أف', 'اوف', 'يخي', 'يزي', 'يلعن', 'تبا', 'لعنة', 'والله']
def interjection_score(text):
    if not isinstance(text, str):
        return 0
    return sum(1 for w in interjections if w in text.lower())

def quotation_score(text):
    if not isinstance(text, str):
        return 0
    count = (text.count('"') + text.count("'") + text.count('«') + text.count('»') +
             text.count('\u201c') + text.count('\u201d'))
    return min(count / (len(text) + 1), 1.0)

def repeated_chars_score(text):
    if not isinstance(text, str):
        return 0
    matches = re.findall(r'(.)\1{2,}', text)
    return min(len(matches) / (len(text) + 1), 1.0)

def tweet_length(text):
    if not isinstance(text, str):
        return 0
    return min(len(text) / 280, 1.0)

intensifiers = ['جدا', 'جداً', 'برشة', 'ياسر', 'للغاية', 'أكيد', 'كثير', 'بزاف']
def intensifier_score(text):
    return sum(1 for word in intensifiers if word in text)

irony_markers = ['طبعاً', 'طبعا', 'أكيد', 'يا سلام', 'ممتاز', 'رائع', 'شكراً', 'شكرا',
                 'بالتوفيق', 'يعطيك العافية']
def irony_score(text):
    return sum(1 for marker in irony_markers if marker in text)

negations = ['موش', 'مش', 'ليس', 'لا', 'ما', 'مو']
positive_words = ['باهي', 'رائع', 'ممتاز', 'حلو', 'زين', 'كويس', 'جميل']
def negated_positive(text):
    words =

In [ ]:
import emoji
import re

def discriminant_vocab_score(text):
    if not isinstance(text, str):
        return 0
    words = tokenize_arabic(text)
    if len(words) == 0:
        return 0
    count = sum(1 for w in words if w in vocab_sarc_moji_dedup)
    return count / len(words)

def hashtag_count(text):
    if not isinstance(text, str):
        return 0
    return len(re.findall(r'#\w+', text))

def emoji_count_feature(text):
    if not isinstance(text, str):
        return 0
    return sum(1 for c in text if c in emoji.EMOJI_DATA)

interjections = ['آه', 'أوه', 'يا إلهي', 'واو', 'يا ربي', 'هههه', 'هاها', 'لول', 'lol',
                 'هيه', 'آآآ', 'أف', 'اوف', 'يخي', 'يزي', 'يلعن', 'تبا', 'لعنة', 'والله']

def interjection_score(text):
    if not isinstance(text, str):
        return 0
    return sum(1 for w in interjections if w in text.lower())

def quotation_score(text):
    if not isinstance(text, str):
        return 0
    count = (text.count('"') + text.count("'") + text.count('«') + text.count('»') +
             text.count('\u201c') + text.count('\u201d'))
    return min(count / (len(text) + 1), 1.0)

def repeated_chars_score(text):
    if not isinstance(text, str):
        return 0
    matches = re.findall(r'(.)\1{2,}', text)
    return min(len(matches) / (len(text) + 1), 1.0)

def tweet_length(text):
    if not isinstance(text, str):
        return 0
    return min(len(text) / 280, 1.0)

intensifiers = ['جدا', 'جداً', 'برشة', 'ياسر', 'للغاية', 'أكيد', 'كثير', 'بزاف']

def intensifier_score(text):
    return sum(1 for word in intensifiers if word in text)

irony_markers = ['طبعاً', 'طبعا', 'أكيد', 'يا سلام', 'ممتاز', 'رائع', 'شكراً', 'شكرا',
                 'بالتوفيق', 'يعطيك العافية']

def irony_score(text):
    return sum(1 for marker in irony_markers if marker in text)

negations = ['موش', 'مش', 'ليس', 'لا', 'ما', 'مو']
positive_words = ['باهي', 'رائع', 'ممتاز', 'حلو', 'زين', 'كويس', 'جميل']

def negated_positive(text):
    words = text.split()
    for i, word in enumerate(words[:-1]):
        if word in negations and words[i+1] in positive_words:
            return 1
    return 0

negative_words = ['خايب', 'سيئ', 'وحش', 'خسارة', 'فاشل', 'بايخ']

def negated_negative(text):
    words = text.split()
    for i, word in enumerate(words[:-1]):
        if word in negations and words[i+1] in negative_words:
            return 1
    return 0

hyperboles = ['أفضل شيء', 'في الكون', 'مليون مرة', 'أموت من', 'في العالم',
              'في التاريخ', 'ما شفت', 'ما في أحسن']

def hyperbole_score(text):
    return sum(1 for h in hyperboles if h in text)

def exclamation_density(text):
    count = text.count('!') + text.count('！')
    return min(count / (len(text) + 1), 1.0)

def question_density(text):
    count = text.count('?') + text.count('؟')
    return min(count / (len(text) + 1), 1.0)

def latin_ratio(text):
    latin_chars = len(re.findall(r'[a-zA-Z]', text))
    return latin_chars / (len(text) + 1)

print("✅ Toutes les fonctions de features redéfinies !")

In [ ]:
#Charger le modèle de sentiment

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
from transformers import pipeline

sentiment_model = pipeline(
    "text-classification",
    model="CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment",
    top_k=None,
    framework="pt"
)

def get_sentiment_features(text):
    try:
        results = sentiment_model(text[:512])[0]
        scores = {r['label']: r['score'] for r in results}
        p_pos = scores.get('positive', 0.0)
        p_neg = scores.get('negative', 0.0)
        p_neu = scores.get('neutral', 0.0)
    except:
        p_pos, p_neg, p_neu = 0.0, 0.0, 0.0
    return p_pos, p_neg, p_neu

print("✅ Modèle de sentiment chargé !")

#Extraction complète des 18 features sur train/test dédupliqués

In [ ]:
from tqdm import tqdm
tqdm.pandas()

def extract_all_features_v2(text):
    if not isinstance(text, str):
        text = ""
    
    p_pos, p_neg, p_neu = get_sentiment_features(text)
    f1 = intensifier_score(text)
    f2 = irony_score(text)
    f3 = negated_positive(text)
    f4 = negated_negative(text)
    f5 = hyperbole_score(text)
    f6 = exclamation_density(text)
    f7 = question_density(text)
    f8 = latin_ratio(text)
    f9  = discriminant_vocab_score(text)
    f10 = hashtag_count(text)
    f11 = emoji_count_feature(text)
    f12 = interjection_score(text)
    f13 = quotation_score(text)
    f14 = repeated_chars_score(text)
    f15 = tweet_length(text)
    
    return [p_pos, p_neg, p_neu, f1, f2, f3, f4, f5, f6, f7, f8,
            f9, f10, f11, f12, f13, f14, f15]

columns_v2 = [
    'p_pos', 'p_neg', 'p_neu', 'intensifier', 'irony', 'neg_pos', 'neg_neg',
    'hyperbole', 'exclamation', 'question', 'latin_ratio',
    'discriminant_vocab', 'hashtag_count', 'emoji_count',
    'interjection', 'quotation', 'repeated_chars', 'tweet_length'
]

print("⏳ Extraction des 18 features sur train (dédupliqué)...")
train_features_dedup = df_train_dedup["Emoji_Text"].progress_apply(extract_all_features_v2)
train_prag_dedup = pd.DataFrame(train_features_dedup.tolist(), columns=columns_v2)

print("⏳ Extraction des 18 features sur test (dédupliqué)...")
test_features_dedup = df_test_dedup["Emoji_Text"].progress_apply(extract_all_features_v2)
test_prag_dedup = pd.DataFrame(test_features_dedup.tolist(), columns=columns_v2)

train_prag_dedup["label"] = df_train_dedup["label"].values
test_prag_dedup["label"] = df_test_dedup["label"].values

train_prag_dedup.to_csv("../outputs_dedup/train_pragmatic_v2.csv", index=False)
test_prag_dedup.to_csv("../outputs_dedup/test_pragmatic_v2.csv", index=False)

print("✅ Features pragmatiques (dédupliquées) sauvegardées !")
print(f"Train : {train_prag_dedup.shape}")
print(f"Test  : {test_prag_dedup.shape}")

In [ ]:
train_prag_dedup = pd.read_csv("../outputs_dedup/train_pragmatic_v2.csv")
print("✅ train_prag_dedup rechargé !")
print(train_prag_dedup.shape)

In [ ]:
print("Vérification de fuite potentielle dans les features pragmatiques :\n")

for col in train_prag_dedup.columns:
    if col != "label":
        correlation = train_prag_dedup[col].corr(train_prag_dedup["label"])
        print(f"{col:25} : corrélation avec label = {correlation:.4f}")

In [ ]:
import emoji as emoji_lib

# Extraire tous les emojis de chaque tweet et voir leur association avec le label
def extraire_emojis(text):
    if not isinstance(text, str):
        return []
    return [c for c in text if c in emoji_lib.EMOJI_DATA]

df_train_dedup["emojis_liste"] = df_train_dedup["Emoji_Text"].apply(extraire_emojis)

# Compter, pour chaque emoji, combien de fois il apparaît dans label=0 vs label=1
from collections import defaultdict
emoji_label_counts = defaultdict(lambda: [0, 0])

for emojis, label in zip(df_train_dedup["emojis_liste"], df_train_dedup["label"]):
    for e in emojis:
        emoji_label_counts[e][label] += 1

# Trier par nombre total d'apparitions, garder les 20 plus fréquents
emoji_stats = []
for e, (count_0, count_1) in emoji_label_counts.items():
    total = count_0 + count_1
    if total >= 20:  # ignorer les emojis trop rares
        pct_ironique = count_1 / total * 100
        emoji_stats.append((e, total, count_0, count_1, pct_ironique))

emoji_stats.sort(key=lambda x: x[1], reverse=True)

print("Emoji | Total | Non-ironique | Ironique | % Ironique")
for e, total, c0, c1, pct in emoji_stats[:20]:
    print(f"{e}   | {total:5} | {c0:5}        | {c1:5}    | {pct:.1f}%")